# 第 2 讲：特征空间的变换——模型如何前向执行

这份 notebook 与 2026 年第二课课件配套。目标不是记住某一种模型，而是建立一个可迁移的观察方式：

> **输入 Tensor → 执行 `forward` → 输出 Tensor**

完成本 notebook 后，你应该能够：

- 用 `shape` 描述一次前向计算；
- 区分矩阵乘、对应元素运算和批量矩阵乘；
- 判断 broadcasting 是否成立，并解释数据如何被重复使用；
- 用 `nn.Parameter` 和 `nn.Module` 实现一个可复用模块；
- 理解 `nn.Linear` 如何处理高维输入，以及多个模块如何组合。

本讲只讨论“模型如何计算”。自动求导、反向传播和参数更新将在下一讲展开。


## 1. 环境与可复现性

本 notebook 只依赖 PyTorch。固定随机种子，便于课堂上比较运行结果。


In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"MPS 可用: {getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available()}")


PyTorch 版本: 2.4.0
CUDA 可用: False
MPS 可用: True


## 2. 先学会读 Tensor

Tensor 不只保存数值，还携带描述计算方式的属性：

- `shape`：数据如何组织；
- `dtype`：每个元素如何表示；
- `device`：数据在哪里计算；
- `requires_grad`：是否需要追踪梯度。

本讲最重要的是 `shape`。读一行张量代码时，先写出输入和输出的 shape，再判断各个维度如何变化。


In [2]:
scalar = torch.tensor(3.14)              # []
vector = torch.tensor([1.0, 2.0, 3.0])   # [3]
matrix = torch.arange(6.0).reshape(2, 3) # [2, 3]
high_dim = torch.randn(2, 3, 4, 5)       # [2, 3, 4, 5]

for name, tensor in {
    "scalar": scalar,
    "vector": vector,
    "matrix": matrix,
    "high_dim": high_dim,
}.items():
    print(
        f"{name:8s} shape={tuple(tensor.shape)!s:14s} "
        f"dtype={tensor.dtype} device={tensor.device} "
        f"requires_grad={tensor.requires_grad}"
    )


scalar   shape=()             dtype=torch.float32 device=cpu requires_grad=False
vector   shape=(3,)           dtype=torch.float32 device=cpu requires_grad=False
matrix   shape=(2, 3)         dtype=torch.float32 device=cpu requires_grad=False
high_dim shape=(2, 3, 4, 5)   dtype=torch.float32 device=cpu requires_grad=False


In [3]:
high_dim.to("mps")

tensor([[[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
          [-1.2345, -0.0431, -1.6047, -0.7521,  1.6487],
          [-0.3925, -1.4036, -0.7279, -0.5594, -0.7688],
          [ 0.7624,  1.6423, -0.1596, -0.4974,  0.4396]],

         [[-0.7581,  1.0783,  0.8008,  1.6806,  1.2791],
          [ 1.2964,  0.6105,  1.3347, -0.2316,  0.0418],
          [-0.2516,  0.8599, -1.3847, -0.8712, -0.2234],
          [ 1.7174,  0.3189, -0.4245,  0.3057, -0.7746]],

         [[-1.5576,  0.9956, -0.8798, -0.6011, -1.2742],
          [ 2.1228, -1.2347, -0.4879, -0.9138, -0.6581],
          [ 0.0780,  0.5258, -0.4880,  1.1914, -0.8140],
          [-0.7360, -1.4032,  0.0360, -0.0635,  0.6756]]],


        [[[-0.0978,  1.8446, -1.1845,  1.3835,  1.4451],
          [ 0.8564,  2.2181,  0.5232,  0.3466, -0.1973],
          [-1.0546,  1.2780, -0.1722,  0.5238,  0.0566],
          [ 0.4263,  0.5750, -0.6417, -2.2064, -0.7508]],

         [[ 0.0109, -0.3387, -1.3407, -0.5854,  0.5362],
          [ 0.5246,  

## 3. 前向过程就是一个对输入张量进行计算的程序

下面的函数虽然很简单，但已经具备前向过程的完整形式：接收 Tensor，依次执行算子，返回新的 Tensor。

$$Y=f_2(f_1(X;\theta_1);\theta_2)$$


In [4]:
def tensor_program(x, weight, bias):
    hidden = x @ weight
    output = hidden + bias
    return output


x = torch.randn(2, 3)       # [B, D]
weight = torch.randn(3, 4)  # [D, E]
bias = torch.randn(4)       # [E]
y = tensor_program(x, weight, bias)

print("x:", tuple(x.shape))
print("weight:", tuple(weight.shape))
print("bias:", tuple(bias.shape))
print("y:", tuple(y.shape))
assert y.shape == (2, 4)


x: (2, 3)
weight: (3, 4)
bias: (4,)
y: (2, 4)


## 4. 线性变换：$Y=XW+b$

若 $X\in\mathbb{R}^{B\times D}$、$W\in\mathbb{R}^{D\times E}$、$b\in\mathbb{R}^{E}$，则输出 $Y\in\mathbb{R}^{B\times E}$。

`@`、`torch.matmul` 和 `Tensor.matmul` 在这个例子中执行同一种矩阵乘法。


In [5]:
B, D, E = 2, 3, 4
X = torch.randn(B, D)
W = torch.randn(D, E)
b = torch.randn(E)

Y_at = X @ W + b
Y_torch = torch.matmul(X, W) + b
Y_method = X.matmul(W) + b

print("X:", tuple(X.shape), "W:", tuple(W.shape), "b:", tuple(b.shape))
print("Y:", tuple(Y_at.shape))
print("三种写法结果一致:", torch.allclose(Y_at, Y_torch) and torch.allclose(Y_at, Y_method))


X: (2, 3) W: (3, 4) b: (4,)
Y: (2, 4)
三种写法结果一致: True


## 5. 高维输入与参数共享

对 $X[B,H,D]$ 和 $W[D,E]$ 执行 `X @ W` 时，PyTorch 会保留前导维度 $B,H$，只对最后一个特征维度 $D$ 做线性变换：

$$[B,H,D]@[D,E]\rightarrow[B,H,E]$$

每个 $(b,h)$ 位置的数据不同，但都使用同一个 $W$。这就是参数共享，而不是为每个位置创建一套参数。


In [6]:
B, H, D, E = 2, 3, 4, 5
X = torch.randn(B, H, D)
W = torch.randn(D, E)
b = torch.randn(E)

Y = X @ W + b
print("X:", tuple(X.shape), "W:", tuple(W.shape), "Y:", tuple(Y.shape))

# 逐位置计算，验证所有位置都复用了同一组 W、b。
Y_by_position = torch.empty(B, H, E)
for batch_index in range(B):
    for position_index in range(H):
        Y_by_position[batch_index, position_index] = (
            X[batch_index, position_index] @ W + b
        )

print("批量计算与逐位置计算一致:", torch.allclose(Y, Y_by_position))
assert Y.shape == (B, H, E)


X: (2, 3, 4) W: (4, 5) Y: (2, 3, 5)
批量计算与逐位置计算一致: True


## 6. 先区分三类常见乘法

- `x @ w`：矩阵乘或批量矩阵乘，要求内部维度匹配；
- `x * scale`：对应元素相乘，shape 相同或可以 broadcasting；
- `torch.bmm(a, b)`：显式的三维批量矩阵乘，每个 batch 独立计算。


In [7]:
# 矩阵乘：[B, D] @ [D, E] -> [B, E]
x = torch.randn(2, 3)
w = torch.randn(3, 4)
y_matmul = x @ w

# 对应元素相乘：[B, D] * [D] -> [B, D]
scale = torch.tensor([1.0, 10.0, 100.0])
y_elementwise = x * scale

# 批量矩阵乘：[B, M, D] @ [B, D, E] -> [B, M, E]
a = torch.randn(2, 3, 4)
b = torch.randn(2, 4, 5)
y_bmm = torch.bmm(a, b)

print("matmul:", tuple(y_matmul.shape))
print("elementwise:", tuple(y_elementwise.shape))
print("bmm:", tuple(y_bmm.shape))


matmul: (2, 4)
elementwise: (2, 3)
bmm: (2, 3, 5)


## 7. 用 `einsum` 显式描述维度关系

`einsum` 给每个维度命名，因此特别适合检查“哪些维度进入输出”。例如：

$$Y_{bhe}=\sum_d X_{bhd}W_{de}$$

对初学者而言，不必先记复杂语法；先读懂输入下标和输出下标即可。


In [8]:
B, H, D, E = 2, 3, 4, 5
X = torch.randn(B, H, D)
W = torch.randn(D, E)

Y_einsum = torch.einsum("bhd,de->bhe", X, W)
Y_matmul = X @ W

A = torch.randn(B, 3, 4)
K = torch.randn(B, 4, 6)
Y_batch_einsum = torch.einsum("bij,bjk->bik", A, K)
Y_batch_bmm = torch.bmm(A, K)

print("einsum 与 matmul 一致:", torch.allclose(Y_einsum, Y_matmul))
print("einsum 与 bmm 一致:", torch.allclose(Y_batch_einsum, Y_batch_bmm))


einsum 与 matmul 一致: True
einsum 与 bmm 一致: True


## 8. Broadcasting：对齐后重复使用

判断两个 shape 能否 broadcasting：

1. 从最后一个维度向前对齐；
2. 较短 Tensor 缺失的前导维度视为 `1`；
3. 对应维度要么相等，要么其中一个为 `1`；
4. 结果的每个维度取两者中较大的尺寸。

大小为 `1` 表示该维只有一份数据，可以沿这一维重复用于多个位置。这里的“重复使用”是逻辑含义，PyTorch 不一定真的复制底层数据。


In [9]:
# 例 1：[B, H, E] + [E]
X = torch.tensor([
    [[1, 2, 3], [4, 5, 6]],
    [[7, 8, 9], [10, 11, 12]],
])                              # [2, 2, 3]
bias = torch.tensor([10, 20, 30])  # [3]，对齐时视为 [1, 1, 3]

Y = X + bias
print("X shape:", tuple(X.shape), "bias shape:", tuple(bias.shape))
print(Y)

expected = torch.tensor([
    [[11, 22, 33], [14, 25, 36]],
    [[17, 28, 39], [20, 31, 42]],
])
assert torch.equal(Y, expected)


X shape: (2, 2, 3) bias shape: (3,)
tensor([[[11, 22, 33],
         [14, 25, 36]],

        [[17, 28, 39],
         [20, 31, 42]]])


In [10]:
# 例 2：[B, H, E] * [H, 1]
scale = torch.tensor([[10], [100]])  # [2, 1]，对齐时视为 [1, 2, 1]

Y = X * scale
print("X shape:", tuple(X.shape), "scale shape:", tuple(scale.shape))
print(Y)

expected = torch.tensor([
    [[10, 20, 30], [400, 500, 600]],
    [[70, 80, 90], [1000, 1100, 1200]],
])
assert torch.equal(Y, expected)


X shape: (2, 2, 3) scale shape: (2, 1)
tensor([[[  10,   20,   30],
         [ 400,  500,  600]],

        [[  70,   80,   90],
         [1000, 1100, 1200]]])


In [11]:
# 一个不能 broadcasting 的例子：[B, H, E] + [D]，其中 E != D
X = torch.randn(2, 3, 4)
incompatible = torch.randn(5)

try:
    X + incompatible
except RuntimeError as error:
    print("预期中的报错:")
    print(error)


预期中的报错:
The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 2


## 9. 参数也是 Tensor，但需要由模型管理

`nn.Parameter` 是一种会被 `nn.Module` 自动注册的 Tensor。注册后，它会：

- 出现在 `model.parameters()` 中；
- 随 `model.to(device)` 一起移动；
- 进入 `state_dict()`，可以保存和加载；
- 在训练时被优化器发现。


In [12]:
class FeatureTransform(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_dim, out_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        return x @ self.weight + self.bias


transform = FeatureTransform(in_dim=4, out_dim=5)
X = torch.randn(2, 3, 4)
Y = transform(X)

print(transform)
print("X:", tuple(X.shape), "Y:", tuple(Y.shape))
print("参数:", [(name, tuple(parameter.shape)) for name, parameter in transform.named_parameters()])
print("state_dict:", {name: tuple(value.shape) for name, value in transform.state_dict().items()})
assert Y.shape == (2, 3, 5)


FeatureTransform()
X: (2, 3, 4) Y: (2, 3, 5)
参数: [('weight', (4, 5)), ('bias', (5,))]
state_dict: {'weight': (4, 5), 'bias': (5,)}


## 10. 使用 `model(x)`，不要直接调用 `model.forward(x)`

`nn.Module.__call__` 会在内部触发 `forward`，同时保留 hooks、编译和框架管理逻辑。两种写法在简单例子中可能得到相同数值，但日常代码应统一写成 `model(x)`。


In [13]:
events = []
handle = transform.register_forward_hook(
    lambda module, inputs, output: events.append(
        (tuple(inputs[0].shape), tuple(output.shape))
    )
)

Y_normal = transform(X)          # 推荐：会经过 Module.__call__，forward hook 生效
print("model(x) 后记录到的 hook:", events)

events.clear()
Y_direct = transform.forward(X)  # 数值相同，但绕过了 __call__ 的框架逻辑
print("model.forward(x) 后记录到的 hook:", events)
print("两种写法数值相同:", torch.allclose(Y_normal, Y_direct))

handle.remove()


model(x) 后记录到的 hook: [((2, 3, 4), (2, 3, 5))]
model.forward(x) 后记录到的 hook: []
两种写法数值相同: True


## 11. `nn.Linear` 的权重布局

课件为了符合数学直觉，把权重写成 $W[D,E]$，计算 `x @ W`。PyTorch 的 `nn.Linear(D, E)` 把权重保存为 `[E,D]`，因此等价的手动计算是：

```python
x @ linear.weight.T + linear.bias
```

Linear 会保留所有前导维度，只把最后一个维度从 `in_features` 变成 `out_features`。


In [14]:
B, H, D, E = 2, 3, 4, 5
linear = nn.Linear(D, E)
X = torch.randn(B, H, D)

Y = linear(X)
Y_manual = X @ linear.weight.T + linear.bias

print("weight:", tuple(linear.weight.shape))
print("bias:", tuple(linear.bias.shape))
print("X:", tuple(X.shape), "Y:", tuple(Y.shape))
print("与手动计算一致:", torch.allclose(Y, Y_manual))
assert Y.shape == (B, H, E)


weight: (5, 4)
bias: (5,)
X: (2, 3, 4) Y: (2, 3, 5)
与手动计算一致: True


## 12. Module 可以组合，但非线性不可缺少

多个 Module 可以像普通函数一样嵌套调用。需要注意：如果两个 Linear 中间没有激活函数，它们整体仍然等价于一个 Linear；加入非线性后，模型才有能力表达更复杂的变换。


In [15]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.up = nn.Linear(dim, hidden_dim)
        self.activation = nn.ReLU()
        self.down = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        hidden = self.activation(self.up(x))
        return self.down(hidden)


mlp = MLP(dim=4, hidden_dim=8)
X = torch.randn(2, 3, 4)
Y = mlp(X)

print(mlp)
print("X:", tuple(X.shape), "Y:", tuple(Y.shape))
assert Y.shape == X.shape


MLP(
  (up): Linear(in_features=4, out_features=8, bias=True)
  (activation): ReLU()
  (down): Linear(in_features=8, out_features=4, bias=True)
)
X: (2, 3, 4) Y: (2, 3, 4)


In [16]:
# 验证：两个不带 bias 的 Linear 连续使用，仍等价于一个 Linear。
first = nn.Linear(4, 6, bias=False)
second = nn.Linear(6, 5, bias=False)
X = torch.randn(2, 3, 4)

Y_two_layers = second(first(X))
effective_weight = second.weight @ first.weight  # [5, 4]
Y_one_layer = X @ effective_weight.T

print("两层 Linear 与合并后的单层一致:", torch.allclose(Y_two_layers, Y_one_layer))


两层 Linear 与合并后的单层一致: True


## 13. 前向过程也可以有分支与残差

`forward` 不必是一条直线。它可以包含分支、复用、条件和残差连接。残差结构要求相加的两个 Tensor shape 兼容。


In [17]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.ffn = MLP(dim, hidden_dim)

    def forward(self, x):
        branch = self.ffn(x)
        return x + branch


block = ResidualBlock(dim=4, hidden_dim=8)
X = torch.randn(2, 3, 4)
Y = block(X)

print("输入:", tuple(X.shape))
print("分支输出:", tuple(block.ffn(X).shape))
print("残差输出:", tuple(Y.shape))
assert Y.shape == X.shape


输入: (2, 3, 4)
分支输出: (2, 3, 4)
残差输出: (2, 3, 4)


## 14. 本讲检查清单与练习

面对一个陌生模型，先逐行回答：

1. 当前 Tensor 的 shape 是什么？每个维度表示什么？
2. 这一行执行什么算子？
3. 哪些维度保持不变，哪些维度发生变化？
4. 参数是否在不同位置被共享？
5. broadcasting 是否符合“右对齐、相等或为 1”的规则？
6. 分支相加或拼接时，shape 是否兼容？

建议练习：

- 修改 broadcasting 两个例子的 `B、H、E`，运行并预测输出 shape；
- 为 `FeatureTransform` 增加一个激活函数，再与 `nn.Sequential` 的结果比较；
- 给 `ResidualBlock` 注册 forward hook，记录每个子模块的输入输出 shape；
- 尝试构造一个 shape 不兼容的残差连接，阅读 PyTorch 的错误信息。

下一讲将在同一个 `nn.Module` 视角下加入 loss、`backward()` 和优化器，回答模型如何从数据中学习。
